In [ ]:
!nvidia-smi

In [ ]:
!pip install --quiet torch

In [ ]:
!pip install --quiet pytorch-lightning
!pip install --quiet torchmetrics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

mypath = '/content/drive/MyDrive/AttackDetectionML'

 # Libraries

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import time
from itertools import product
from os.path import join as pjoin
from pandas import read_pickle as rpckl
from pandas import read_csv as rcsv
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score

import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torchmetrics import MeanSquaredError, R2Score

from multiprocessing import cpu_count
import random

import shutil


In [ ]:
random.seed(42)
pl.seed_everything(42)  # Reproduce thing


# Parameters

## Datasets

In [ ]:
net = 'CH'
ds_type = 'injection'
seq = 24

save = True

## Path

In [ ]:
data_folder = pjoin(mypath, 'raw_data')
ds_path = pjoin(mypath, 'datasets', net)
res_dir = pjoin(mypath, 'results', 'unsupervised', 'single_node_attack', net)

print(f'data_folder:\n\t{data_folder}\n')
print(f'ds_path:\n\t{ds_path}\n')
print(f'res_dir:\n\t{res_dir}\n')

## Model

In [ ]:
N_EPOCHS = 50
BATCH_SIZE = 128
DROPOUT = 0.75
LEARNING_RATE = 0.0001
N_HIDDEN = 256
N_LAYERS = 3

# Functions

# Class

## Data Module

In [ ]:
class AnomalyDataModule(pl.LightningDataModule):

  def __init__(self, train_ds, val_ds, test_ds, batch_size):
    super().__init__()
    self.train_dataset = train_ds
    self.val_dataset = val_ds
    self.test_dataset = test_ds
    self.batch_size = batch_size

  def train_dataloader(self):
    return DataLoader(
        self.train_dataset,
        batch_size=self.batch_size,
        shuffle=True,
        num_workers=cpu_count()
        )

  def val_dataloader(self):
    return DataLoader(
        self.val_dataset,
        batch_size=self.batch_size,
        shuffle=False,
        num_workers=cpu_count()
        )

  def test_dataloader(self):
    return DataLoader(
        self.test_dataset,
        batch_size=self.batch_size,
        shuffle=False,
        num_workers=cpu_count()
        )

## Model

In [ ]:
class SequenceModel(nn.Module):
  def __init__(self, n_features, DROPOUT,
               n_hidden=N_HIDDEN, n_layers=N_LAYERS):
    super().__init__()

    self.lstm = nn.LSTM(
        input_size=n_features,
        hidden_size=n_hidden,
        num_layers=n_layers,
        batch_first=True,
        dropout=DROPOUT,
    )

    self.regressor = nn.Linear(n_hidden, 1)

  def forward(self, x):
    self.lstm.flatten_parameters()
    _, (hidden, _) = self.lstm(x)

    out = hidden[-1]
    return self.regressor(out)

## Predictor

In [ ]:
class AnomalyPredictor(pl.LightningModule):

  def __init__(self, n_features: int):
    super().__init__()
    self.model = SequenceModel(n_features, DROPOUT)

    self.criterion = nn.MSELoss()
    self.rmse_scorer = MeanSquaredError(squared=False)
    self.r2_scorer = R2Score()

  def forward(self, x, labels=None):
    output = self.model(x)
    loss = 0
    if labels is not None:
      loss = self.criterion(output, labels)
    return loss, output

  def training_step(self, batch, batch_idx):
    sequences, labels = batch
    loss, outputs = self(sequences, labels)
    # predictions = torch.argmax(outputs, dim=1)
    predictions = outputs
    step_rmse_score = self.rmse_scorer(predictions, labels)
    step_r2_score = self.r2_scorer(predictions, labels)

    self.log('train_loss', loss, prog_bar=True, logger=True)
    self.log('train_rmse_score', step_rmse_score, prog_bar=True, logger=True)
    self.log('train_r2_score', step_r2_score, prog_bar=True, logger=True)
    return {'loss': loss, 'rmse_score': step_rmse_score, 'r2_score': step_r2_score}

  def validation_step(self, batch, batch_idx):
    sequences, labels = batch
    loss, outputs = self(sequences, labels)
    # predictions = torch.argmax(outputs, dim=1)
    predictions = outputs
    step_rmse_score = self.rmse_scorer(predictions, labels)
    step_r2_score = self.r2_scorer(predictions, labels)

    self.log('val_loss', loss, prog_bar=True, logger=True)
    self.log('val_rmse_score', step_rmse_score, prog_bar=True, logger=True)
    self.log('val_r2_score', step_r2_score, prog_bar=True, logger=True)
    return {'loss': loss, 'rmse_score': step_rmse_score, 'r2_score': step_r2_score}

  def test_step(self, batch, batch_idx):
    sequences, labels = batch
    loss, outputs = self(sequences, labels)
    # predictions = torch.argmax(outputs, dim=1)
    predictions = outputs
    step_rmse_score = self.rmse_scorer(predictions, labels)
    step_r2_score = self.r2_scorer(predictions, labels)

    self.log('test_loss', loss, prog_bar=True, logger=True)
    self.log('test_rmse_score', step_rmse_score, prog_bar=True, logger=True)
    self.log('test_r2_score', step_r2_score, prog_bar=True, logger=True)
    return {'loss': loss, 'rmse_score': step_rmse_score, 'r2_score': step_r2_score}

  def configure_optimizers(self):
    return optim.Adam(self.parameters(), lr=LEARNING_RATE)

# Load data


## Features

In [ ]:
def load_data(country_code, n_year=20):

    gen_info = rcsv(pjoin(data_folder, 'gens_info.csv'))
    gen_info = gen_info[gen_info.country==country_code]


    # DOWNLOAD GEN FIRST YEAR
    year = 2016
    index = 1
    print(f'>>> importing gen data: year {year}, series {index}')
    gen = rcsv(pjoin(data_folder, f'gens_{year}_{index}.csv'))
    gen.columns = gen.columns.astype(int)
    gen = gen[gen_info.id]
    gen *= 100  # per units /!\

    # DOWNLOAD GEN OTHER YEARS
    for i in range(n_year - 1):  # DOWNLOAD AND ADD N-1 OTHERS YEARS
        year += 1
        if year > 2020:
            index += 1
            year = 2016
        if index > 4:
            break

        print(f'>>> importing gen data: year {year}, series {index}')
        df = rcsv(pjoin(data_folder, f'gens_{year}_{index}.csv'))
        df.columns = df.columns.astype(int)
        df = df[gen_info.id]
        df *= 100  # per units /!\

        gen = pd.concat([gen, df], ignore_index=True)

    if ds_type == 'generation':
        load = []
    else:
        load_info = rcsv(pjoin(data_folder, 'loads_info.csv'))
        load_info = load_info[load_info.country==country_code]


        # DOWNLOAD LOAD FIRST YEAR
        year = 2016
        index = 1
        print(f'>>> importing load data: year {year}, series {index}')
        load = rcsv(pjoin(data_folder, f'loads_{year}_{index}.csv'))
        load.columns = load.columns.astype(int)
        load = load[load_info.id]
        load *= 100  # per units /!\

        # DOWNLOAD LOAD OTHER YEARS
        for i in range(n_year - 1):  # DOWNLOAD AND ADD N-1 OTHERS YEARS
            year += 1
            if year > 2020:
                index += 1
                year = 2016
            if index > 4:
                break

            print(f'>>> importing load data: year {year}, series {index}')
            df = rcsv(pjoin(data_folder, f'loads_{year}_{index}.csv'))
            df.columns = df.columns.astype(int)
            df = df[load_info.id]
            df *= 100

            load = pd.concat([load, df], ignore_index=True)

        # Add suffix to load to avoid duplicated in injection dataset
        load.columns = load.columns.astype(str)
        load = load.add_suffix('_load')

    return gen, load

In [ ]:
gen_p, load_p = load_data(net)

classification_test_index = rpckl(pjoin(ds_path, 'test_timesteps.p'))[0].to_list()

regression_validation_index = rpckl(pjoin(ds_path, 'regression_validation_timesteps.p')).to_list()
regression_train_index = rpckl(pjoin(ds_path, 'regression_train_timesteps.p')).to_list()

# Avoid printing unwanted logs

In [ ]:
# Remove prompt that appear every times

"""
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
WARNING:pytorch_lightning.loggers.tensorboard:Missing logger folder: lightning_logs/Sils_gen_generation
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.
"""

import logging
logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.WARNING)
logging.getLogger("pytorch_lightning.accelerators.cuda").setLevel(logging.WARNING)
logging.getLogger("pytorch_lightning.loggers.tensorboard").setLevel(logging.ERROR)

# LOOP FOR ALL NODES


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
attacked_gens = rpckl(pjoin(ds_path, 'attacked_gens.p')).to_list()
# attacked_gens = [173]  # DEBUG CH
# attacked_gens = [253]  # DEBUG DE
# attacked_gens = [1011]  # DEBUG ES


start_time = time.time()  # For total running time
fit_time = start_time
for attacked_gen in attacked_gens:
  print(f'\n{attacked_gen} - {ds_type} - {seq} - {time.time()-fit_time:.1f} seconds')
  fit_time = time.time()  # For fit running time


  ## Path manager
  dir_path = pjoin(res_dir,
                   'lstm',
                   ds_type,
                   f'{attacked_gen}',
                   f'sequence_len-{seq}',
                   f'contextual_hist',
                    )
  os.makedirs(dir_path, exist_ok=True)

  ## LOAD DATASET
  X = gen_p.copy()

  if ds_type == 'injection':
      X = pd.concat([X, load_p], axis=1)
  y = X[attacked_gen].to_frame()

  attacked_gen_id = X.columns.get_loc(attacked_gen)

  ## SCALER
  X.columns = X.columns.astype(str)
  X_scaler = MinMaxScaler()
  X = X_scaler.fit_transform(X)
  y_scaler = MinMaxScaler()
  y = y_scaler.fit_transform(y)

  X = X.reshape((20, 8736, -1))

  # roll the column of the attacked generator by one hour
  X[:, :, attacked_gen_id] = np.roll(X[:, :, attacked_gen_id], 1, axis=1)

  # put together the features with their history
  X = np.expand_dims(X, 2)
  X = np.concatenate([np.roll(X, seq - t, axis=1) for t in range(seq + 1)], axis=2)
  X = X.reshape((20*8736, seq+1, -1))

  # split dataset
  X_train = X[regression_train_index]
  y_train = y[regression_train_index]

  X_val = X[regression_validation_index]
  y_val = y[regression_validation_index]

  X_test = X[classification_test_index]
  y_test = y[classification_test_index]

  ## LOAD TRAIN, VAL & TEST DATASET
  train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
  val_ds = TensorDataset(torch.tensor(X_val).float(), torch.tensor(y_val).float())
  test_ds = TensorDataset(torch.tensor(X_test).float(), torch.tensor(y_test).float())

  ## Data Module
  data_module = AnomalyDataModule(train_ds, val_ds, test_ds, BATCH_SIZE)

  ## Model
  model = AnomalyPredictor(n_features = X_train.shape[-1])


  ## Checkpoint & Logger
  param_str = f'Seq-{seq}ts_Hid-{N_HIDDEN}_Lay-{N_LAYERS}'
  checkpoint_callback = ModelCheckpoint(
                            dirpath=f'checkpoints/{attacked_gen}/{ds_type}',
                            filename=f'best_{param_str}',
                            save_top_k=1,
                            verbose=False,  # No print best score for epochs
                            # verbose=True,
                            monitor='val_loss',
                            mode='min',
                                      )

  logs_file_name = f'{attacked_gen}_{ds_type[:3]}_{param_str}'
  logger = TensorBoardLogger('lightning_logs',
                             name=logs_file_name)


  ## Trainer
  trainer = pl.Trainer(
                logger=logger,
                callbacks=[checkpoint_callback],
                max_epochs=N_EPOCHS,
                accelerator='gpu',
                devices=1,
                enable_progress_bar=False,  # No progress bar
                enable_model_summary=False,  # No model summary
                # enable_progress_bar=True,
                # enable_model_summary=True,
                      )


  ## Training
  trainer.fit(model, data_module)


  ## Save lightning_logs
  dst_folder = pjoin(mypath, 'unsupervised', 'lightning_logs',
                     net, logs_file_name)
  if os.path.exists(dst_folder): shutil.rmtree(dst_folder)
  shutil.copytree(
        pjoin('/content', 'lightning_logs', logs_file_name),
        dst_folder,)


  ## Save best model
  shutil.copy2(
      trainer.checkpoint_callback.best_model_path,
      pjoin(dir_path, 'best_checkpoint.ckpt'),
      )


  ## Load trained model
  trained_model = AnomalyPredictor.load_from_checkpoint(
                              trainer.checkpoint_callback.best_model_path,
                              n_features=X_train.shape[-1],
                                                      )

  trained_model.freeze()  # just for inference, disable dropout and gradient calculation

  trained_model = trained_model.to(device)


  ## Validation prediction
  _, y_val_predict = trained_model(torch.tensor(X_val).float().to(device))
  p_val_predict = y_scaler.inverse_transform(np.array(y_val_predict.to('cpu'))).squeeze()
  p_val =y_scaler.inverse_transform(y_val).squeeze()

  ## Test prediction
  _, y_test_predict = trained_model(torch.tensor(X_test).float().to(device))
  p_test_predict = y_scaler.inverse_transform(np.array(y_test_predict.to('cpu'))).squeeze()
  p_test =y_scaler.inverse_transform(y_test).squeeze()

  ## Metrics
  r2_val = r2_score(p_val, p_val_predict)
  rmse_val = root_mean_squared_error(p_val, p_val_predict)

  r2_test = r2_score(p_test, p_test_predict)
  rmse_test = root_mean_squared_error(p_test, p_test_predict)


  ## Save
  metrics = pd.DataFrame({
            'r\u00b2': [r2_val, r2_test],
            'rmse': [rmse_val, rmse_test],
            },
            index = ['validation set', 'test set'],)


  ## Print results
  print(metrics.round(3))


  ## SAVE METRICS
  metrics.to_csv(pjoin(dir_path, 'gscv_regression_metrics.csv'))


  ## SAVE PREDICTIONS
  pd.DataFrame({
      'y_val': p_val,
      'y_val_predict': p_val_predict,
      }, index=regression_validation_index).to_csv(pjoin(dir_path, 'prediction_validation_set.csv'))

  pd.DataFrame({
      'y_test': p_test,
      'y_test_predict': p_test_predict,
      }, index=classification_test_index).to_csv(pjoin(dir_path, 'prediction_test_set.csv'))


ex_time = int(time.time() - start_time)
print(f"\nTotal run time :\t{ex_time} [s]  -  {ex_time/3600:.2f} [h] ")

In [ ]:
from google.colab import runtime
runtime.unassign()  # disconnect and delete the runtime environment

## Model description


In [ ]:
# from prettytable import PrettyTable
# def count_parameters(model):
#     table = PrettyTable(["Modules", "Parameters"])
#     total_params = 0
#     for name, parameter in model.named_parameters():
#         if not parameter.requires_grad: continue
#         params = parameter.numel()
#         table.add_row([name, params])
#         total_params+=params
#     print(table)
#     print(f"Total Trainable Params: {total_params}")
#     return total_params

# count_parameters(model)

In [ ]:
# !pip install --quiet torchinfo

In [ ]:
# import torchinfo

# torchinfo.summary(model,
#                   # (64, 4, 199),
#                   # batch_dim=0,
#                   # col_names = ("input_size", "output_size", "num_params", "kernel_size", "mult_adds"),
#                   )